#  🐍📊 NOTEBOOK DE TIMEO 🐍📊

## === GUIDE des fonctions du module local datalake === 

from DATALAKE.data import *

**data = data_download_gmd("country")** -> Telechargement des données liés à un pays via global_macro_dataset (BD publique)
**data = data_download_fred("indicator"** : str, "start" : str , "end" : str) -> Telechargement d'un indicateur via la FRED (BD publique)

**data_storing(data : dataframe, "nom_fichier" : str)** -> Range un dataframe "data" dans le DATALAKE en parquet, que vous venez de télécharger d'internet 

**data = import_parquet("file_name" : str)** -> importe un dataframe du DATALAKE dans votre file cible (notebook ou .py)

**which_parquet()** -> Vous renvoie une liste de l'ensemble des parquets dispo dans le DATALAKE

## === Pour importer les fonctions du fichier "outils_eda", si vous en avez besoin lors de votre analyse ===

import sys
import os

sys.path.append("../dorian_code")

import outils_eda

In [ ]:
import sys
import os

# On remonte d'un dossier (on sort de timeo_code pour aller dans CODE_ZONE)
# Et on pointe vers le dossier dorian_code
sys.path.append("../dorian_code")

# Maintenant tu peux l'importer normalement
import outils_eda

In [ ]:
from DATALAKE.data import *

In [87]:
data = import_parquet('main_cleaned')
data.columns = [a.strip().replace(' ','_').lower() for a in data.columns]

In [88]:
# data_storing(data, 'main_cleaned')
display(data)

,year,gdp_nominal,expected_inflation,taux_changes,cpi,taux_directeur,export,import,yield_perpetual,oil_price,gdp_cycle,gdp_trend,output_gap
0,1850.0,5.461955e+02,-4.263878,4.870213,1.086743,3.00,5840.612037,4074.867428,3.134833,NaN,-9.394881,5.555904e+02,-59.137567
1,1851.0,5.603683e+02,-2.100840,4.912750,1.054140,3.00,6186.571635,4360.390146,3.112000,NaN,-22.648580,5.830169e+02,-25.741873
2,1852.0,5.815871e+02,1.373391,4.901500,1.054140,2.00,6505.397147,4229.966188,3.031167,NaN,-28.762305,6.103494e+02,-21.220463
3,1853.0,6.552916e+02,15.664691,4.887750,1.152175,5.00,7489.007769,4994.885074,3.101583,NaN,18.024024,6.372676e+02,35.356566
4,1854.0,7.175112e+02,7.906296,4.882500,1.326154,5.00,7448.306639,4822.161455,3.299917,NaN,54.347907,6.631633e+02,12.202187
...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,2022.0,2.270145e+06,0.026000,1.370000,112.000000,0.25,661210.000000,684210.000000,0.850000,51.76,-102188.081351,2.372333e+06,-23.215360
173,2023.0,2.504231e+06,0.091000,1.240000,122.200000,3.50,823450.000000,895120.000000,2.370000,81.40,37232.291453,2.466999e+06,66.259653
174,2024.0,2.685410e+06,0.073000,1.240000,131.100000,5.25,875200.000000,902300.000000,4.050000,66.52,121291.899735,2.564118e+06,21.140061
175,2025.0,2.745120e+06,0.028000,1.270000,134.800000,5.25,889140.000000,898450.000000,4.150000,64.65,83333.054515,2.661787e+06,31.941550


## **CREATION DE L'OUTPUT GAP NATUREL - Hodrick Prescott Filter (HP)** 

#### **Principe**
Dans la construction de l'ouput gap nous partons d'une hypothèse simple. Le PIB fluctue autour d'une trend dynamique à travers des mouvements conjoncturels. On peut dans ce acdre assimilés les mouvements réalisés par le PIB observé autour de la trend à des bruits. L'objectif en utilisant le filtre de Hodrick-Prescott nous permet d'arriver à nos fins en "dé-noisant" le bruit afin de mettre en avant la tendance cyclique du PIB. Il nous permettra ici dans ce contexte de séparé le GDP en une composante "trend" et une componsante conjoncturecyclique "conjoncturelle" en calculant gdp_trend (PIB conjoncturel observé) et gdp_cycle constituant l'output naturel.

#### **Décomposition du PIB**
Le PIB réel $Y_t$ peut être décomposé en une tendance (PIB potentiel) $Y^*_t$ et un composant cyclique $c_t$ :

$$
Y_t = Y^*_t + c_t
$$

#### **Minimisation avec filtre HP**
Le PIB potentiel $Y^*_t$ est obtenu en résolvant le problème suivant :

$$
\min_{\{Y^*_t\}} \sum_{t=1}^{T} (Y_t - Y^*_t)^2 
+ \lambda \sum_{t=2}^{T-1} \big( (Y^*_{t+1} - Y^*_t) - (Y^*_t - Y^*_{t-1}) \big)^2
$$

où $\lambda$ est le paramètre de lissage (typiquement $\lambda = 100$ pour un PIB trimestriel). Le $\lambda$ représente l'intensité de la filtration, plus le lambda est élevé, plus la tendance sera lisse et inversement. Nous adoptons lambda = 100 comme une convention macro pour lisser le cycle annuel.

#### **Implémentation en Python** :



In [89]:
import pandas as pd 
from statsmodels.tsa.filters.hp_filter import hpfilter
import plotly.graph_objects as go

# filter et creation des deux agregats 
gdp_clean = data["gdp_nominal"].ffill()
gdp_cycle, gdp_trend = hpfilter(gdp_clean, lamb=100)

In [90]:
# Ajout des deux colonnes au df
if "gdp_cycle" and "gdp_trend" and "output_gap" not in data.columns:
    data["gdp_cycle"] = gdp_cycle
    data["gdp_trend"] = gdp_trend
    data["output_gap"] = (data["gdp_nominal"] - data["gdp_cycle"])/data["gdp_cycle"]
    print ("gdp_cycle and gdp_trend have been added to the datas")
else:
    print ("gdp_cycle and gdp_trend already in the datas")

data_storing(data, "main")



gdp_cycle and gdp_trend already in the datas


In [91]:
# Plot des deux composantes 
# plot de l'output gap
# Plot des deux composantes 
# plot de l'output gap
fig_outputgap = go.Figure([
        go.Scatter(x = data['year'],
                   y = data["output_gap"],
                   line = dict(color = "blue"),
                   name = "trend"
        )
])

# plot des composantes du HP filter
fig_comp_hp = go.Figure([
        go.Scatter(x = data['year'],
                   y = data["gdp_trend"],
                   line = dict(color = "blue"),
                   name = "trend"
        ),

        go.Scatter(x = data['year'],
                   y = data["gdp_cycle"],
                   line = dict(color = 'green'),
                   name = "cycle"
        )
    ])
fig_outputgap.show()
fig_comp_hp.show()



